In [2]:
%pip install langchain-community
%pip install unstructured
%pip install pdfminer
%pip install pi-heif
%pip uninstall -y pdfminer.six
%pip install pdfminer.six>=20231228
%pip install --upgrade unstructured[pdf]

Note: you may need to restart the kernel to use updated packages.
Note: you may need to restart the kernel to use updated packages.
Note: you may need to restart the kernel to use updated packages.
Note: you may need to restart the kernel to use updated packages.
Found existing installation: pdfminer.six 20251107
Uninstalling pdfminer.six-20251107:
  Successfully uninstalled pdfminer.six-20251107
Note: you may need to restart the kernel to use updated packages.
Note: you may need to restart the kernel to use updated packages.
  Using cached google_cloud_vision-3.11.0-py3-none-any.whl.metadata (9.8 kB)
  Using cached effdet-0.4.1-py3-none-any.whl.metadata (33 kB)
  Using cached google_api_core-2.28.1-py3-none-any.whl.metadata (3.3 kB)
Using cached effdet-0.4.1-py3-none-any.whl (112 kB)
Using cached google_cloud_vision-3.11.0-py3-none-any.whl (529 kB)
Using cached google_api_core-2.28.1-py3-none-any.whl (173 kB)
Note: you may need to restart the kernel to use updated packages.


In [1]:
from langchain_ollama import ChatOllama

llm_tc = ChatOllama(
    base_url="http://localhost:11434",
    model="gpt-oss:20b-cloud",
    temperature=0.5
)

llm = ChatOllama(
    base_url="http://localhost:11434",
    model="ministral-3:8b-cloud",
    temperature=0.5
)

/mnt/e/LocalFlow/.venv/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
import os
from langchain_community.document_loaders import (
UnstructuredPDFLoader
) 
doc_folder = './docs'
documents = []   
for filename in os.listdir(doc_folder):
        file_path = os.path.join(doc_folder, filename)
        loader = UnstructuredPDFLoader(file_path)
        docs = loader.load();
        for doc in docs:
            documents.append(f"{filename}\n{doc.page_content.strip()}")
requirements_txt = "\n\n".join(documents[:3][:1000])  # Limit to first 1000 characters of first 3 docs

        

Could get FontBBox from font descriptor because None cannot be parsed as 4 floats
Could get FontBBox from font descriptor because None cannot be parsed as 4 floats
Could get FontBBox from font descriptor because None cannot be parsed as 4 floats
Could get FontBBox from font descriptor because None cannot be parsed as 4 floats
Could get FontBBox from font descriptor because None cannot be parsed as 4 floats
Could get FontBBox from font descriptor because None cannot be parsed as 4 floats
Could get FontBBox from font descriptor because None cannot be parsed as 4 floats
Could get FontBBox from font descriptor because None cannot be parsed as 4 floats
Could get FontBBox from font descriptor because None cannot be parsed as 4 floats
Could get FontBBox from font descriptor because None cannot be parsed as 4 floats
Could get FontBBox from font descriptor because None cannot be parsed as 4 floats
Could get FontBBox from font descriptor because None cannot be parsed as 4 floats
Could get FontBB

In [3]:
from langchain.tools import tool 
from langchain_core.prompts import PromptTemplate

@tool
def generate_TestCase(user_story: str) -> str:
    """Generate a test case based on the given user story."""
    prompt_template = PromptTemplate.from_template(
        """
        You are a software testing expert. 
        Given the following user story, generate a detailed test case that includes the test title, description, preconditions, steps to execute, expected results, and postconditions.
        include the combinations of valid invalid edge cases and alternative flow scenarios 
                
        user story:{user_story}
        Format the test case in a clear and structured manner.
        Format:
        Test Case Title: <Title>
        Description: <Description>
        Preconditions: <Preconditions>
        Steps to Execute:
        1. <Step 1>
        2. <Step 2>
        ...Expected Results:
        1. <Expected Result 1>
        2. <Expected Result 2>
        ...Postconditions: <Postconditions> 
        and ensure clarity and completeness.  

         no gerkin syntax
        """
    )
    #prompt = prompt_template.invoke(user_story=user_story)
    prompt = prompt_template.invoke({"user_story": user_story})
    return llm_tc.invoke(prompt)
    #response = llm.generate([prompt.format(user_story=user_story)])
    #return response.generations[0][0].text.strip()

@tool
def generate_TestCase_formPDF() -> str:
    """Generate a test case based on the given user story."""
    prompt_template = PromptTemplate.from_template(
        """
        You are a software testing expert. 
        Given the following user story, generate a detailed test case that includes the test title, description, preconditions, steps to execute, expected results, and postconditions.
        include the combinations of valid invalid edge cases and alternative flow scenarios 
                
        Requirements_txt :{requirements_txt}
        Format the test case in a clear and structured manner.
        Format:
        Test Case Title: <Title>
        Description: <Description>
        Preconditions: <Preconditions>
        Steps to Execute:
        1. <Step 1>
        2. <Step 2>
        ...Expected Results:
        1. <Expected Result 1>
        2. <Expected Result 2>
        ...Postconditions: <Postconditions> 
        and ensure clarity and completeness.  

         no gerkin syntax
        """
    )
    #prompt = prompt_template.invoke(user_story=user_story)
    prompt = prompt_template.invoke({"requirements_txt": requirements_txt})
    return llm_tc.invoke(prompt)
    #response = llm.generate([prompt.format(user_story=user_story)])
    #return response.generations[0][0].text.strip()


In [4]:
import os
from langchain.agents import create_agent
agent = create_agent(
    model=llm,
    tools=[generate_TestCase_formPDF]
)

UserStory_input = """As a user, I want to be able to reset my password so that I can regain access to my account if I forget my current password."""

#result = agent.invoke(f"Generate a BDD test case for the following user story: {UserStory_input}")
#print(result)
# 2. Invoke with a dictionary containing "messages"
result = agent.invoke({
    "messages": [("user", f"Generate a  test case for the following user story: {UserStory_input}")]
})

# 3. Print the output (usually found in the last message)
print(result["messages"][-1].content)

Here’s a structured **test case** for the user story:
**"As a user, I want to be able to reset my password so that I can regain access to my account if I forget my current password."**

---

### **Test Case Title: Password Reset Functionality**

#### **Description:**
This test case validates the end-to-end password reset workflow, ensuring users can securely regain access to their accounts when they forget their password. It covers normal flows, edge cases, and error scenarios to guarantee a seamless and secure experience.

---

### **Preconditions:**
1. A test user account exists with valid email registration.
2. The user’s email address is verified and active.
3. The password reset functionality (APIs/services) is operational.
4. Email delivery services (e.g., SMTP) are functional.
5. The system clock is synchronized.

---

### **Steps to Execute:**

#### **1. Successful Password Reset Flow**
| Step | Action | Expected Result |
|------|--------|------------------|
| 1.1 | User naviga

Other optional items are 

  title 
        preconditions
        Action steps 
        expected results
        clean up steps  

result = agent.invoke({
    "messages": [("user", f"Generate a  BDD test case for the following user story: {UserStory_input}")]
})